In [1]:
import pandas as pd
from datetime import datetime as dt
from datetime import date

from kiblib.utils.db import DbConn

In [2]:
db_conn = DbConn().create_engine()

In [60]:
query = """SELECT i.itemnumber, i.biblionumber, i.biblioitemnumber, i.barcode, i.dateaccessioned, i.homebranch, i.notforloan, i.itemcallnumber, i.location, i.ccode, bi.itemtype, b.title, b.author
FROM koha_prod.items i
JOIN koha_prod.biblioitems bi ON bi.biblionumber = i.biblionumber
JOIN koha_prod.biblio b ON b.biblionumber = i.biblionumber
WHERE bi.itemtype = 'PE' AND i.notforloan = '0'"""

items = pd.read_sql(query,db_conn)
len(items)

5651

In [75]:
perios = items.groupby(['biblionumber', 'title', 'author', 'homebranch', 'ccode'])['itemnumber'].count().reset_index()

In [76]:
query = """SELECT biblionumber, itemnumber, issue_id, borrowernumber, branch
FROM statdb.stat_issues
WHERE itemtype = 'PE'
AND DATE(issuedate) >= CURDATE() - INTERVAL 1 YEAR"""

prets = pd.read_sql(query,db_conn)
len(prets)

17476

In [77]:
prets_titre_nb = prets.groupby(['biblionumber', 'branch'])['issue_id'].count().reset_index()
prets_titre_emprunteurs_distincts = prets.groupby(['biblionumber', 'branch'])['borrowernumber'].nunique().reset_index()

In [78]:
perios = perios.merge(prets_titre_nb, left_on=['biblionumber', 'homebranch'], right_on=['biblionumber', 'branch'], how='left')
perios = perios[['biblionumber', 'title', 'author', 'homebranch', 'ccode', 'itemnumber', 'issue_id']]

In [79]:
perios = perios.merge(prets_titre_emprunteurs_distincts, left_on=['biblionumber', 'homebranch'], right_on=['biblionumber', 'branch'], how='left')
perios = perios[['biblionumber', 'title', 'author', 'homebranch', 'ccode', 'itemnumber', 'issue_id', 'borrowernumber']]

In [80]:
query = """SELECT av.authorised_value,av.lib 
FROM koha_prod.authorised_values av
WHERE category = 'collection'"""
va_collection = pd.read_sql(query, db_conn)

In [81]:
#perios = perios[perios['itemnumber'] > 1]
perios = perios.merge(va_collection, left_on='ccode', right_on='authorised_value', how='left')
perios

,biblionumber,title,author,homebranch,ccode,itemnumber,issue_id,borrowernumber,authorised_value,lib
0,154602,Spirou,réd. en chef Patrick Pinchart,MED,JPRZZZZ,58,113.0,20.0,NaN,NaN
1,154603,Alternatives économiques,dir. de publ. Denis Clerc,MED,ACFPAZZ,18,67.0,35.0,ACFPAZZ,ACF - Presse d'actualité
2,154605,Archéologia,dir. de publ. Louis Faton,MED,ACFHSZZ,17,7.0,7.0,ACFHSZZ,ACF - Histoire
3,154607,Astrapi,dir. de la rédaction Pascal Ruffenach,BUS,JPRZZZZ,23,139.0,28.0,NaN,NaN
4,154607,Astrapi,dir. de la rédaction Pascal Ruffenach,MED,JPRZZZZ,46,263.0,65.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
393,376436,Politis,réd. en chef Christophe Kantcheff,MED,P17,18,2.0,2.0,P17,P17 - PERIODIQUE
394,376437,Futu&r,dir. de pub. Jérôme Ruskin,MED,P17,1,1.0,1.0,P17,P17 - PERIODIQUE
395,376438,L'écologiste,dir. de publ. Thierry Jaccaud,MED,P17,1,NaN,NaN,P17,P17 - PERIODIQUE
396,376439,Plénior,dir. de publ. Pascal Birenzweigue,MED,P17,2,1.0,1.0,P17,P17 - PERIODIQUE


In [82]:
perios = perios[['biblionumber', 'title', 'author', 'ccode', 'lib', 'homebranch', 'itemnumber',
       'issue_id', 'borrowernumber']]

In [83]:
perios.columns = ['notice', 'titre', 'auteur', 'ccode', 'collection', 'site', 'nb exemplaires',
       'nb prêts', 'emprunteurs distincts']

In [84]:
perios['nb prêts'] = perios['nb prêts'].fillna(0)
perios['nb prêts'] = perios['nb prêts'].astype(int)

perios['emprunteurs distincts'] = perios['emprunteurs distincts'].fillna(0)
perios['emprunteurs distincts'] = perios['emprunteurs distincts'].astype(int)

/tmp/ipykernel_29338/753565272.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perios['nb prêts'] = perios['nb prêts'].fillna(0)
/tmp/ipykernel_29338/753565272.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  perios['nb prêts'] = perios['nb prêts'].astype(int)
/tmp/ipykernel_29338/753565272.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydat

In [85]:
perios.loc[perios['ccode'] == 'JPRZZZZ', 'collection'] = 'Jeunesse - presse'
perios.loc[perios['ccode'] == 'AMVZZ', 'collection'] = 'AMV - Musique Généralités'
perios = perios.sort_values(by='ccode')
perios = perios[['notice', 'titre', 'auteur', 'collection', 'site', 'nb exemplaires',
       'nb prêts', 'emprunteurs distincts']]

In [86]:
perios.to_excel("perios.xlsx", index=False)